# Bayesian Model Comparison: Prior Sensitivity and Predictive Performance

**Author:** Dipon Konar  
**Keywords:** Bayesian Inference, Model Selection, Cross-Validation, Marginal Likelihood

---

## Executive Summary

This project demonstrates comprehensive Bayesian model comparison techniques using two competing models with different prior assumptions. We employ multiple evaluation metrics—including information-theoretic measures and cross-validation—to assess both in-sample fit and predictive performance on held-out data. The analysis reveals how prior specification significantly impacts posterior inference and model predictions, with implications for principled prior selection in Bayesian workflows.

## 1. Problem Statement

### Context
We observe binary outcome counts from 10 independent trials where each trial consists of 20 Bernoulli experiments. We aim to estimate the underlying probability of success (θ) and compare two competing Bayesian models that encode different prior beliefs about this parameter.

### Observed Data
```
y = [10, 15, 15, 14, 14, 14, 13, 11, 12, 16]
```
Each value represents the number of successes in 20 trials.

### Research Questions
1. How do different prior specifications affect posterior inference about θ?
2. Which model better explains the observed training data (in-sample fit)?
3. Which model generalizes better to new, unseen data (out-of-sample performance)?
4. How sensitive are model comparisons to prior specification?
5. How do marginal likelihoods differ across prior specifications?

## 2. Bayesian Model Specifications

### Model 1: Weakly Informative Prior
$$y_i \\sim \\text{Binomial}(n=20, \\theta)$$
$$\\theta \\sim \\text{Beta}(6, 6)$$

**Interpretation:** Beta(6,6) is symmetric and centered at 0.5, expressing weak prior knowledge favoring moderate success probabilities while remaining relatively diffuse.

### Model 2: Informative Prior
$$y_i \\sim \\text{Binomial}(n=20, \\theta)$$
$$\\theta \\sim \\text{Beta}(20, 60)$$

**Interpretation:** Beta(20,60) is heavily skewed toward lower values (mean ≈ 0.25), encoding strong prior belief that the true success probability is substantially below 0.5.

### Likelihood Function
For a single observation $y_i$:
$$p(y_i|\\theta) = \\binom{20}{y_i} \\theta^{y_i} (1-\\theta)^{20-y_i}$$

### Posterior (Analytical)
Due to conjugacy between Binomial likelihood and Beta prior:
$$\\theta|y \\sim \\text{Beta}(a + \\sum y_i, b + 20n - \\sum y_i)$$

## 3. Setup and Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.special import beta as beta_function, logsumexp

# Set random seed for reproducibility
np.random.seed(42)

# Configure visualization defaults
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'legend.fontsize': 10
})

# Observed training data
y_train = np.array([10, 15, 15, 14, 14, 14, 13, 11, 12, 16])
n_trials = 20  # Trials per observation
n_observations = len(y_train)
total_successes = np.sum(y_train)
total_failures = n_observations * n_trials - total_successes

print(f"Training Data Summary:")
print(f"  Observations: {y_train}")
print(f"  N observations: {n_observations}")
print(f"  Total successes: {total_successes}")
print(f"  Total failures: {total_failures}")
print(f"  Sample proportion: {total_successes / (n_observations * n_trials):.4f}")

## 4. Prior Assumptions and Justification

### Prior 1: Beta(6, 6)
- **Prior Mean:** E[θ] = 6/(6+6) = 0.50
- **Prior Variance:** Var[θ] ≈ 0.0038
- **Interpretation:** Symmetric around 0.5; weakly informative; implies ~12 equivalent observations

### Prior 2: Beta(20, 60)
- **Prior Mean:** E[θ] = 20/(20+60) = 0.25
- **Prior Variance:** Var[θ] ≈ 0.0028
- **Interpretation:** Strongly informative; heavily skewed toward low success probability; implies ~80 equivalent observations

### Why Compare These Priors?
This comparison demonstrates the tension between data and prior knowledge:
- **Model 1** relies primarily on observed data (weak prior)
- **Model 2** encodes strong skepticism about high success rates

In [ ]:
# Define prior specifications
priors = {
    'Model 1 (Weak Prior)': {'a': 6, 'b': 6},
    'Model 2 (Strong Prior)': {'a': 20, 'b': 60}
}

# Calculate prior statistics
prior_stats = {}
for model_name, params in priors.items():
    a, b = params['a'], params['b']
    mean = a / (a + b)
    var = (a * b) / ((a + b)**2 * (a + b + 1))
    prior_stats[model_name] = {'mean': mean, 'var': var, 'std': np.sqrt(var)}

prior_df = pd.DataFrame(prior_stats).T
print("\nPrior Distribution Statistics:")
print(prior_df.round(6))

## 5. Methodology

### 5.1 Posterior Inference
Using the Beta-Binomial conjugate family, we derive analytical posteriors.

### 5.2 In-Sample Evaluation: Deviance
$$\\text{In-sample Deviance} = -2 \\times \\text{LPPD}$$
$$\\text{LPPD} = \\sum_{i=1}^{n} \\log \\frac{1}{S} \\sum_{s=1}^{S} p(y_i|\\theta^{(s)})$$

Lower deviance = better fit to training data.

### 5.3 Out-of-Sample Evaluation
Evaluate on held-out test set:
$$\\text{OOS Deviance} = -2 \\times \\sum_{i \\in \\text{test}} \\log \\frac{1}{S} \\sum_{s=1}^{S} p(y_i|\\theta^{(s)})$$

### 5.4 Leave-One-Out Cross-Validation
1. For each observation, fit model on remaining n-1 observations
2. Evaluate predictive density on held-out observation
3. Repeat for all observations
4. LOO-CV Deviance = -2 × sum of log predictive densities

### 5.5 Marginal Likelihood
$$p(y) = \\int p(y|\\theta) p(\\theta) d\\theta$$

Estimated via Monte Carlo: $p(y) \\approx \\frac{1}{S} \\sum_{s=1}^{S} p(y|\\theta^{(s)}), \\quad \\theta^{(s)} \\sim p(\\theta)$

## 6. Implementation: Core Functions

In [ ]:
def posterior_params(y_data, a_prior, b_prior, n_trials=20):
    """
    Compute posterior Beta parameters using conjugacy.
    Returns: (a_posterior, b_posterior)
    """
    k_total = np.sum(y_data)
    n_total = len(y_data) * n_trials
    a_post = a_prior + k_total
    b_post = b_prior + (n_total - k_total)
    return a_post, b_post


def compute_lppd(y_data, a_prior, b_prior, n_trials=20, n_samples=10000):
    """
    Compute Log Pointwise Predictive Density using posterior samples.
    """
    a_post, b_post = posterior_params(y_data, a_prior, b_prior, n_trials)
    theta_samples = np.random.beta(a_post, b_post, n_samples)
    
    lppd_values = []
    for yi in y_data:
        likelihoods = stats.binom.pmf(yi, n_trials, theta_samples)
        avg_likelihood = np.mean(likelihoods)
        lppd_values.append(np.log(avg_likelihood))
    
    return np.sum(lppd_values)


def loo_cross_validation(y_data, a_prior, b_prior, n_trials=20, n_samples=10000):
    """
    Perform leave-one-out cross-validation.
    """
    fold_scores = []
    
    for i in range(len(y_data)):
        y_train = np.delete(y_data, i)
        y_test = y_data[i]
        
        a_post, b_post = posterior_params(y_train, a_prior, b_prior, n_trials)
        theta_samples = np.random.beta(a_post, b_post, n_samples)
        
        likelihoods = stats.binom.pmf(y_test, n_trials, theta_samples)
        log_pred_dens = np.log(np.mean(likelihoods))
        fold_scores.append(log_pred_dens)
    
    loo_deviance = -2 * np.sum(fold_scores)
    return {'loo_deviance': loo_deviance, 'fold_scores': fold_scores}


def marginal_likelihood_mc(y_data, a_prior, b_prior, n_trials=20, n_samples=100000):
    """
    Estimate marginal likelihood via Monte Carlo integration.
    """
    theta_samples = np.random.beta(a_prior, b_prior, n_samples)
    log_likelihoods = np.zeros(n_samples)
    
    for yi in y_data:
        log_likelihoods += stats.binom.logpmf(yi, n_trials, theta_samples)
    
    return np.exp(logsumexp(log_likelihoods) - np.log(n_samples))


print("Core functions defined successfully.")

## 7. Posterior Estimation

In [ ]:
# Compute posterior parameters for both models
posterior_results = {}

for model_name, prior_params in priors.items():
    a_post, b_post = posterior_params(
        y_train, prior_params['a'], prior_params['b'], n_trials
    )
    
    post_mean = a_post / (a_post + b_post)
    post_var = (a_post * b_post) / ((a_post + b_post)**2 * (a_post + b_post + 1))
    post_std = np.sqrt(post_var)
    
    ci_lower = stats.beta.ppf(0.025, a_post, b_post)
    ci_upper = stats.beta.ppf(0.975, a_post, b_post)
    
    posterior_results[model_name] = {
        'a_post': a_post, 'b_post': b_post, 'mean': post_mean,
        'std': post_std, 'ci_lower': ci_lower, 'ci_upper': ci_upper
    }

posterior_summary = pd.DataFrame({
    'Model': list(posterior_results.keys()),
    'α (posterior)': [posterior_results[m]['a_post'] for m in posterior_results],
    'β (posterior)': [posterior_results[m]['b_post'] for m in posterior_results],
    'Mean': [posterior_results[m]['mean'] for m in posterior_results],
    'Std Dev': [posterior_results[m]['std'] for m in posterior_results],
    '95% CI Lower': [posterior_results[m]['ci_lower'] for m in posterior_results],
    '95% CI Upper': [posterior_results[m]['ci_upper'] for m in posterior_results]
})

print("\nPosterior Distribution Summary:")
print(posterior_summary.to_string(index=False))
print("\nSample mean:", total_successes / (n_observations * n_trials))

## 8. Visualization: Priors and Posteriors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

theta_seq = np.linspace(0, 1, 1000)
colors = ['#1f77b4', '#ff7f0e']
model_names = list(posterior_results.keys())
sample_mean = total_successes / (n_observations * n_trials)

for idx, (model_name, color) in enumerate(zip(model_names, colors)):
    result = posterior_results[model_name]
    prior_params = priors[model_name]
    
    prior_dens = stats.beta.pdf(theta_seq, prior_params['a'], prior_params['b'])
    post_dens = stats.beta.pdf(theta_seq, result['a_post'], result['b_post'])
    
    axes[0].plot(theta_seq, prior_dens, linestyle='--', color=color, linewidth=2, label=model_name, alpha=0.7)
    axes[1].plot(theta_seq, post_dens, color=color, linewidth=2.5, label=model_name)
    axes[1].axvline(result['mean'], color=color, linestyle=':', linewidth=2, alpha=0.6)
    axes[1].fill_between(theta_seq, 0, post_dens, color=color, alpha=0.15)

axes[0].axvline(sample_mean, color='green', linestyle='-', linewidth=2, label=f'Sample Mean = {sample_mean:.3f}', alpha=0.8)
axes[1].axvline(sample_mean, color='green', linestyle='-', linewidth=2, label=f'Sample Mean = {sample_mean:.3f}', alpha=0.8)

axes[0].set_xlabel('θ (Success Probability)', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Prior Distributions', fontsize=13, fontweight='bold')
axes[0].legend(loc='upper right', fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('θ (Success Probability)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Posterior Distributions', fontsize=13, fontweight='bold')
axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Prior and posterior distributions visualized.")

## 9. In-Sample Model Evaluation

In [ ]:
in_sample_results = {}

for model_name, prior_params in priors.items():
    lppd = compute_lppd(y_train, prior_params['a'], prior_params['b'], n_trials)
    deviance = -2 * lppd
    in_sample_results[model_name] = {'lppd': lppd, 'deviance': deviance}

print("\n" + "="*70)
print("IN-SAMPLE MODEL EVALUATION (Training Data)")
print("="*70)

for model_name, results in in_sample_results.items():
    print(f"\n{model_name}:")
    print(f"  Log Pointwise Predictive Density: {results['lppd']:.4f}")
    print(f"  In-Sample Deviance:               {results['deviance']:.4f}")

deviance_diff = in_sample_results['Model 2 (Strong Prior)']['deviance'] - in_sample_results['Model 1 (Weak Prior)']['deviance']
print(f"\nDeviance Difference (M2 - M1): {deviance_diff:+.4f}")
print(f"Better In-Sample Fit: Model 1 (Weak Prior)" if deviance_diff > 0 else "Better In-Sample Fit: Model 2")

## 10. Out-of-Sample Evaluation

In [ ]:
y_test = np.array([5, 6, 10, 8, 9])

print("\nTest Data:")
print(f"  Values: {y_test}")
print(f"  Mean: {np.mean(y_test):.2f}")
print(f"  Sample proportion: {np.sum(y_test) / (len(y_test) * n_trials):.4f}")

oos_results = {}

for model_name, prior_params in priors.items():
    a_post, b_post = posterior_params(y_train, prior_params['a'], prior_params['b'], n_trials)
    theta_samples = np.random.beta(a_post, b_post, 10000)
    
    lppd_test = 0
    for yi in y_test:
        likelihoods = stats.binom.pmf(yi, n_trials, theta_samples)
        lppd_test += np.log(np.mean(likelihoods))
    
    deviance_test = -2 * lppd_test
    oos_results[model_name] = {'lppd': lppd_test, 'deviance': deviance_test}

print("\n" + "="*70)
print("OUT-OF-SAMPLE MODEL EVALUATION (Test Data)")
print("="*70)

for model_name, results in oos_results.items():
    print(f"\n{model_name}:")
    print(f"  Log Pointwise Predictive Density: {results['lppd']:.4f}")
    print(f"  Out-of-Sample Deviance:           {results['deviance']:.4f}")

deviance_diff_oos = oos_results['Model 2 (Strong Prior)']['deviance'] - oos_results['Model 1 (Weak Prior)']['deviance']
print(f"\nDeviance Difference (M2 - M1): {deviance_diff_oos:+.4f}")

## 11. Leave-One-Out Cross-Validation

In [ ]:
print("\n" + "="*70)
print("LEAVE-ONE-OUT CROSS-VALIDATION")
print("="*70)

loo_results = {}

for model_name, prior_params in priors.items():
    loo_cv = loo_cross_validation(y_train, prior_params['a'], prior_params['b'], n_trials)
    loo_results[model_name] = loo_cv
    
    print(f"\n{model_name}:")
    print(f"  LOO-CV Deviance: {loo_cv['loo_deviance']:.4f}")
    print(f"  Mean Fold Score: {np.mean(loo_cv['fold_scores']):.4f}")

loo_diff = loo_results['Model 2 (Strong Prior)']['loo_deviance'] - loo_results['Model 1 (Weak Prior)']['loo_deviance']
print(f"\nLOO Deviance Difference (M2 - M1): {loo_diff:+.4f}")
print(f"Better Generalization: Model 1 (Weak Prior)" if loo_diff > 0 else "Better Generalization: Model 2")

## 12. Prior Sensitivity: Marginal Likelihood

In [ ]:
print("\n" + "="*80)
print("PRIOR SENSITIVITY: MARGINAL LIKELIHOOD")
print("="*80)

prior_configs = [
    ('Beta(0.1, 0.4)', 0.1, 0.4),
    ('Beta(1, 1)', 1.0, 1.0),
    ('Beta(2, 6)', 2.0, 6.0),
    ('Beta(6, 2)', 6.0, 2.0),
    ('Beta(20, 60)', 20.0, 60.0),
    ('Beta(60, 20)', 60.0, 20.0)
]

# Test on simple data: k=2, n=10
y_simple = np.array([2])
print("\nMarginal Likelihoods for k=2 successes in n=10 trials:")

ml_results = []
for prior_name, a, b in prior_configs:
    ml_mc = marginal_likelihood_mc(y_simple, a, b, n_trials=10, n_samples=100000)
    ml_results.append({'Prior': prior_name, 'ML': ml_mc, 'Log ML': np.log(ml_mc)})

ml_df = pd.DataFrame(ml_results)
print(ml_df.to_string(index=False))
best_idx = ml_df['ML'].idxmax()
print(f"\nBest Prior: {ml_df.loc[best_idx, 'Prior']}")

# Training data marginal likelihood
print("\n" + "-"*80)
print("\nMarginal Likelihoods for Training Data:")
ml_train = []
for model_name, prior_params in priors.items():
    a, b = prior_params['a'], prior_params['b']
    ml = marginal_likelihood_mc(y_train, a, b, n_trials, n_samples=100000)
    ml_train.append({
        'Model': model_name,
        'Marginal Likelihood': f"{ml:.6e}",
        'Log ML': f"{np.log(ml):.6f}"
    })

ml_df_train = pd.DataFrame(ml_train)
print(ml_df_train.to_string(index=False))

## 13. Comprehensive Model Comparison

In [ ]:
comparison_data = []
for model_name in priors.keys():
    comparison_data.append({
        'Model': model_name,
        'In-Sample Dev': f"{in_sample_results[model_name]['deviance']:.4f}",
        'OOS Dev': f"{oos_results[model_name]['deviance']:.4f}",
        'LOO-CV Dev': f"{loo_results[model_name]['loo_deviance']:.4f}",
        'Post. Mean': f"{posterior_results[model_name]['mean']:.4f}",
        'Post. SD': f"{posterior_results[model_name]['std']:.6f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*100)
print("COMPREHENSIVE MODEL COMPARISON SUMMARY")
print("="*100)
print(comparison_df.to_string(index=False))
print("\nNote: Lower deviance values indicate better model performance.")

## 14. Key Findings and Recommendations

### Summary of Results

**1. Prior Specification Impact**
- Model 1's weak prior (Beta(6,6)) allows the posterior to follow the data closely
- Model 2's strong prior (Beta(20,60)) pulls the posterior away from observed data
- When prior beliefs conflict with strong evidence, posterior inference deteriorates

**2. Model Performance Metrics**
- Model 1 achieves superior in-sample fit (lower deviance)
- Model 1 demonstrates better generalization by LOO-CV
- Model 2's OOS advantage is coincidental (test data happened to be lower-valued)

**3. Marginal Likelihood Insights**
- Model 1 has higher marginal likelihood, indicating better prior-data compatibility
- Supports Bayesian model comparison via Bayes factors

### Recommendation

**Use Model 1 (Beta(6,6) prior) because:**
1. ✓ Superior in-sample fit
2. ✓ Better LOO-CV performance (robust generalization)
3. ✓ Higher marginal likelihood
4. ✓ Well-calibrated posterior
5. ✓ Weak prior lets data speak for itself

### Best Practices
1. Always perform prior sensitivity analysis
2. Use multiple evaluation metrics (in-sample, OOS, CV)
3. Report credible intervals and posterior uncertainty
4. Check for prior-data conflict
5. For small samples (n<20), prefer LOO-CV over external test splits

## 15. Reproducibility

In [ ]:
import sys

print("\n" + "="*80)
print("REPRODUCIBILITY & SESSION INFORMATION")
print("="*80)
print(f"""
Python Version: {sys.version.split()[0]}
NumPy Version: {np.__version__}
SciPy Version: {stats.scipy.__version__}
Pandas Version: {pd.__version__}
Matplotlib Version: {plt.matplotlib.__version__}

Random Seed: 42 (set at notebook start)

Key Hyperparameters:
  • Posterior samples for LPPD: 10,000
  • MC marginal likelihood samples: 100,000
  • Credible interval coverage: 95%
  • LOO-CV folds: {len(y_train)}

This analysis is fully reproducible with seed=42
""")